# CQL2 Search, Collection Search & Geoid Lookup

**Personas**
- **GS — Geospatial Scientist**: queries items using spatial/temporal/attribute predicates, paginates large result sets.
- **SI — System Integrator**: wires collection discovery and geoid-based asset resolution to downstream tools (e.g. TiTiler tile server).

## Structure

| Part | Topic |
|------|-------|
| 1 | CQL2 Spatial + Temporal + Attribute Search with pagination |
| 2 | Full-text Collection Search across catalogs |
| 3 | Geoid-based Lookup for TiTiler integration |

## Prerequisites

- A running GeoID / DynaStore instance.
- A `.env` file (or exported shell variables) containing:
  - `DYNASTORE_BASE_URL` — e.g. `http://localhost:8080`
  - `DYNASTORE_TOKEN` — bearer token
  - `CATALOG_ID` — target catalog identifier
  - `COLLECTION_ID` — target collection identifier
- `pip install httpx python-dotenv`

In [ ]:
import os
import json
import uuid
from dotenv import load_dotenv
import httpx

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN         = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
# Create a notebook-unique catalog so the notebook is fully self-contained
_CATALOG_SUPPLIED = os.environ.get("CATALOG_ID", "")
CATALOG_ID    = _CATALOG_SUPPLIED or f"qry-test-{uuid.uuid4().hex[:8]}"
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")
_OWN_CATALOG  = not bool(_CATALOG_SUPPLIED)  # track if we created it

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"Base URL    : {BASE_URL}")
print(f"Auth        : {'present' if TOKEN else 'absent'}")
print(f"Catalog     : {CATALOG_ID}  (own={_OWN_CATALOG})")
print(f"Collection  : {COLLECTION_ID}")

if _OWN_CATALOG:
    # Provision a fresh catalog + collection so the notebook is self-contained
    r = client.post("/stac/catalogs", content=json.dumps({
        "id": CATALOG_ID, "type": "Catalog", "title": "CQL2 Search Test",
        "description": "Ephemeral catalog for queryables notebook.", "stac_version": "1.0.0",
    }))
    assert r.status_code == 201, f"Catalog create failed: {r.status_code}: {r.text}"

    r = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk", content=json.dumps({"configs": {
        "CollectionPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {"enabled": True, "operations": {
            "WRITE": [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            "READ":  [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        }},
    }}), timeout=60.0)
    print(f"Catalog defaults: {r.status_code}")

    r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", content=json.dumps({
        "id": COLLECTION_ID, "type": "Collection", "stac_version": "1.0.0",
        "title": "Sentinel-2 L2A Test", "description": "Test collection.",
        "license": "proprietary",
        "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
        "links": [],
    }))
    assert r.status_code == 201, f"Collection create failed: {r.status_code}: {r.text}"
    print(f"Created catalog '{CATALOG_ID}' + collection '{COLLECTION_ID}'")

## Part 1 — CQL2 Spatial + Temporal + Attribute Search

GeoID exposes a STAC-compliant `/stac/catalogs/{catalog_id}/search` endpoint that accepts both **CQL2-JSON** (POST body `filter` key) and **CQL2-Text** (GET query parameter `filter`).  
Filters are evaluated by the pygeofilter pipeline before reaching the storage driver.

The region of interest below covers the **Horn of Africa** — a primary FAO crop-monitoring domain for dekadal NDVI and rainfall products.

In [ ]:
# --- STAC POST search: bbox + datetime + limit ---
search_body = {
    "bbox": [33.0, 3.0, 51.5, 22.0],          # Horn of Africa
    "datetime": "2020-01-01T00:00:00Z/..",
    "limit": 10,
}

resp = client.post(f"/stac/catalogs/{CATALOG_ID}/search", json=search_body)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
assert "features" in data, "Response must contain a 'features' key"

returned = data.get("context", {}).get("returned", len(data["features"]))
print(f"Returned    : {returned} items")
print(f"Total matched (context): {data.get('context', {}).get('matched', 'n/a')}")
for feat in data["features"][:3]:
    props = feat.get("properties", {})
    print(f"  id={feat['id']}  datetime={props.get('datetime', '?')}")

In [ ]:
# --- OGC Features GET items: same collection via GET ---
params_get = {
    "bbox": "33.0,3.0,51.5,22.0",
    "datetime": "2020-01-01T00:00:00Z/..",
    "limit": 10,
}

resp_get = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params=params_get,
)
assert resp_get.status_code == 200, f"OGC items GET failed: {resp_get.status_code}: {resp_get.text}"

data_get = resp_get.json()
# OGC Features returns FeatureCollection; STAC search also returns FeatureCollection
features_key = "features" if "features" in data_get else "items"
assert features_key in data_get, f"Response must contain 'features' or 'items': {list(data_get.keys())}"

returned_get = len(data_get[features_key])
print(f"OGC GET items returned: {returned_get} items")
print("OGC Features GET items endpoint works.")

In [ ]:
# --- Pagination: follow next link ---
links = data.get("links", [])
next_link = next((lnk for lnk in links if lnk.get("rel") == "next"), None)

if next_link is None:
    print("No next page — result set fits in one page, skipping pagination check.")
else:
    href = next_link["href"]
    if href.startswith("http"):
        resp_p2 = httpx.get(href, headers=headers, timeout=120.0)
    else:
        resp_p2 = client.get(href)

    assert resp_p2.status_code == 200, f"Page 2 request failed: {resp_p2.status_code}: {resp_p2.text}"
    data_p2 = resp_p2.json()
    assert "features" in data_p2, "Page 2 response must contain a 'features' key"

    ids_p1 = {f["id"] for f in data["features"]}
    ids_p2 = {f["id"] for f in data_p2["features"]}
    assert ids_p1.isdisjoint(ids_p2), (
        f"Page 1 and Page 2 share item IDs — pagination is broken.\nOverlap: {ids_p1 & ids_p2}"
    )
    print(f"Page 1 items: {len(ids_p1)}, Page 2 items: {len(ids_p2)} — pagination works.")

## Part 2 — Full-text Collection Search

GeoID provides two collection search endpoints:

| Endpoint | Scope |
|----------|-------|
| `POST /stac/collections-search` | All catalogs the user can read |
| `GET /search/catalogs/{catalog_id}/collections-search` | Single catalog |

Both accept a `q` query parameter for free-text matching against collection title/description.

In [ ]:
# --- Catalog collections listing ---
resp_cols = client.get(f"/stac/catalogs/{CATALOG_ID}/collections")
assert resp_cols.status_code == 200, f"Collections listing failed: {resp_cols.status_code}: {resp_cols.text}"

cols_data = resp_cols.json()
collections = (
    cols_data if isinstance(cols_data, list)
    else cols_data.get("collections", [])
)
print(f"Found {len(collections)} collection(s) in catalog '{CATALOG_ID}':")
for col in collections:
    print(f"  [{col.get('id', '?')}] {col.get('title', '?')}")

In [ ]:
# --- STAC collection search (ES-powered, may be unavailable on fresh catalogs) ---
resp_search = client.get(
    f"/search/catalogs/{CATALOG_ID}/collections-search",
    params={"q": ""}
)
print(f"Collection search status: {resp_search.status_code}")
if resp_search.status_code == 200:
    cs_data = resp_search.json()
    all_cols = cs_data.get("collections", cs_data if isinstance(cs_data, list) else [])
    print(f"Collection search returned {len(all_cols)} collection(s).")
else:
    # 500/503 is expected if ES hasn't indexed this catalog yet
    print(f"NOTE: ES collection search returned {resp_search.status_code} — ES may not have indexed this catalog yet.")

In [ ]:
# --- STAC collection detail ---
resp_col = client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
assert resp_col.status_code == 200, f"Collection GET failed: {resp_col.status_code}: {resp_col.text}"

col = resp_col.json()
assert col.get("id") == COLLECTION_ID, f"Unexpected collection id: {col.get('id')}"
print(f"Collection '{COLLECTION_ID}' exists — title: {col.get('title', '?')}")

## Part 3 — Geoid-based Lookup (TiTiler integration)

A **geoid** is GeoID's compact spatial identifier for a geographic cell (geohash-derived).  
The lookup endpoint resolves which collections cover a given geoid cell — enabling TiTiler to discover tile-able assets without a full STAC search.

Endpoint: `GET /search/catalogs/{catalog_id}/geoid/{geoid}`

In [ ]:
# --- Happy path: geoid lookup ---
# A geohash covering Addis Ababa (~9N 38E), precision 4
GEOID = os.environ.get("GEOID", "s174")

resp_geo = client.get(f"/search/catalogs/{CATALOG_ID}/geoid/{GEOID}")
print(f"Geoid lookup status: {resp_geo.status_code}")
# 200 = found; 404 = no items indexed at this geoid; both are valid data states
assert resp_geo.status_code in (200, 404), (
    f"Unexpected status on geoid lookup: {resp_geo.status_code}: {resp_geo.text}"
)
if resp_geo.status_code == 200:
    geo_data = resp_geo.json()
    collections_at_geoid = geo_data.get("collections", [])
    print(f"Geoid '{GEOID}' covered by {len(collections_at_geoid)} collection(s):")
    for col in collections_at_geoid:
        print(f"  {col}")
else:
    print(f"Geoid '{GEOID}' has no indexed items yet (empty catalog).")

In [ ]:
# --- Error cases ---

# 1. Non-existent catalog — geoid endpoint may return 200 (empty list) or 4xx
resp_no_cat = client.get(f"/search/catalogs/__nonexistent__/geoid/{GEOID}")
print(f"Unknown catalog → HTTP {resp_no_cat.status_code}")
assert resp_no_cat.status_code < 500, (
    f"Unexpected 5xx on unknown catalog: {resp_no_cat.status_code}: {resp_no_cat.text}"
)

# 2. Extra params alongside geoid — verify no 5xx
resp_extra = client.get(
    f"/search/catalogs/{CATALOG_ID}/geoid/{GEOID}",
    params={"extra_param": "ignored"}
)
print(f"Geoid + extra param → HTTP {resp_extra.status_code}")
assert resp_extra.status_code < 500, (
    f"Unexpected 5xx on geoid+extra param: {resp_extra.status_code}: {resp_extra.text}"
)

## Driver Capability Check

Some storage drivers do not implement `Capability.SEARCH`.  
When the driver does not expose search, GeoID returns **HTTP 501 Not Implemented** rather than silently returning empty results.

In [ ]:
# --- Collection queryables ---
resp_q = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables"
)
assert resp_q.status_code == 200, f"Queryables failed: {resp_q.status_code}: {resp_q.text}"

q_data = resp_q.json()
properties = q_data.get("properties", {})
queryable_names = sorted(properties.keys())
print(f"Queryable properties for '{COLLECTION_ID}' ({len(queryable_names)}):")
for name in queryable_names:
    prop = properties[name]
    print(f"  {name}: type={prop.get('type', '?')} title={prop.get('title', '?')}")

# Teardown: delete the catalog if this notebook created it
if _OWN_CATALOG:
    r_del = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
    print(f"\nTeardown: DELETE /stac/catalogs/{CATALOG_ID} → {r_del.status_code}")
    assert r_del.status_code == 204, f"Catalog delete failed: {r_del.status_code}: {r_del.text}"

client.close()
print("Session closed.")